# IA não supervisionada: segmentação de clientes

Este notebook mostra um exemplo simples de IA não supervisionada usando `scikit-learn`.

A ideia é encontrar grupos de clientes parecidos, sem uma coluna pronta dizendo qual é o segmento de cada um.

## Problema

Fato: a base não tem rótulo de segmento.

Inferência: depois que o K-Means encontra os grupos, nós interpretamos os centroides e damos nomes aos perfis.

Opinião técnica: K-Means é bom para este exemplo porque é direto, visual e muito usado para explicar agrupamento.

Esta célula prepara o ambiente do notebook de segmentação. Ela configura `LOKY_MAX_CPU_COUNT` para evitar um aviso do `joblib` no Windows, importa `pandas` para tabelas, `plotly` para gráficos e componentes do `scikit-learn` para agrupamento, normalização, PCA e avaliação.

In [1]:
import os

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')

import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_columns', None)

Esta célula lê uma base fictícia de clientes para o problema de segmentação. Cada cliente tem informações de comportamento, como compras por mês, gasto médio e dias desde a última compra. Diferente do exemplo de crédito, aqui não existe uma coluna dizendo qual é o segmento correto. Esse detalhe é o ponto central da IA não supervisionada: o algoritmo precisa descobrir grupos parecidos sem receber uma resposta pronta.

In [2]:
df = pd.read_csv('../data/dados_clientes_nao_supervisionado.csv')

Esta célula valida a base de clientes antes do agrupamento. Ela verifica se há valores nulos, se as colunas usadas são numéricas, se existem valores negativos indevidos e se há possíveis outliers pelo método IQR. Essa validação é importante porque K-Means também é sensível a dados problemáticos, principalmente valores extremos. Na apresentação, vale dizer que a qualidade dos dados influencia diretamente a qualidade dos grupos encontrados.

In [3]:
def validar_base_clientes(base_clientes: pd.DataFrame) -> dict:
    colunas_entrada = ['compras_mes', 'gasto_medio_reais', 'dias_desde_ultima_compra']

    if base_clientes.empty:
        raise ValueError('A base de clientes não pode estar vazia.')

    valores_nulos = base_clientes.isna().sum().to_dict()
    if any(quantidade > 0 for quantidade in valores_nulos.values()):
        raise ValueError(f'Foram encontrados valores nulos: {valores_nulos}')

    for coluna in colunas_entrada:
        if not pd.api.types.is_numeric_dtype(base_clientes[coluna]):
            raise TypeError(f'A coluna {coluna} precisa ser numérica.')

        if (base_clientes[coluna] < 0).any():
            raise ValueError(f'A coluna {coluna} possui valor negativo.')

    outliers_iqr = {}
    for coluna in colunas_entrada:
        primeiro_quartil = base_clientes[coluna].quantile(0.25)
        terceiro_quartil = base_clientes[coluna].quantile(0.75)
        intervalo_iqr = terceiro_quartil - primeiro_quartil
        limite_inferior = primeiro_quartil - 1.5 * intervalo_iqr
        limite_superior = terceiro_quartil + 1.5 * intervalo_iqr
        outliers_iqr[coluna] = int(((base_clientes[coluna] < limite_inferior) | (base_clientes[coluna] > limite_superior)).sum())

    return {
        'quantidade_linhas': len(base_clientes),
        'valores_nulos': valores_nulos,
        'outliers_iqr': outliers_iqr,
    }

Esta célula cria a base de clientes, executa a validação e mostra os dados para inspeção. O objetivo é garantir que a análise começa com uma visão clara das variáveis usadas pelo modelo. Como não existe rótulo de segmento, essa tabela ajuda a entender quais características o algoritmo vai usar para comparar os clientes. É uma etapa simples, mas deixa a explicação mais transparente.

In [4]:
base_clientes = df
resumo_validacao_clientes = validar_base_clientes(base_clientes)

display(base_clientes)
resumo_validacao_clientes

,cliente,compras_mes,gasto_medio_reais,dias_desde_ultima_compra
0,C01,2,35,90
1,C02,1,20,120
2,C03,3,45,80
3,C04,8,180,20
4,C05,10,220,14
...,...,...,...,...
195,C196,10,241,20
196,C197,9,210,33
197,C198,5,124,67
198,C199,11,264,2


{'quantidade_linhas': 200,
 'valores_nulos': {'cliente': 0,
  'compras_mes': 0,
  'gasto_medio_reais': 0,
  'dias_desde_ultima_compra': 0},
 'outliers_iqr': {'compras_mes': 0,
  'gasto_medio_reais': 0,
  'dias_desde_ultima_compra': 1}}

Esta célula cria um boxplot para visualizar possíveis outliers nas variáveis usadas na segmentação de clientes. A validação anterior calcula a quantidade de outliers pelo método IQR, mas o gráfico mostra a distribuição de forma mais fácil de explicar. Aqui os pontos individuais foram ocultados para manter o gráfico mais limpo, mostrando apenas caixa, mediana e bigodes de cada variável. Como K-Means usa distância, valores muito fora do padrão podem puxar os centroides e distorcer os grupos.

In [5]:
colunas_numericas_clientes = ['compras_mes', 'gasto_medio_reais', 'dias_desde_ultima_compra']

dados_boxplot_clientes = base_clientes.melt(
    value_vars=colunas_numericas_clientes,
    var_name='variavel',
    value_name='valor',
)

figura_boxplot_clientes = px.box(
    dados_boxplot_clientes,
    x='variavel',
    y='valor',
    color='variavel',
    points='outliers',
    facet_col='variavel',
    facet_col_wrap=2,
    title='Boxplot das variáveis da base de clientes',
)
figura_boxplot_clientes.update_yaxes(matches=None, showticklabels=True)
figura_boxplot_clientes.update_xaxes(matches=None)
figura_boxplot_clientes.update_layout(template='plotly_white', showlegend=False)
figura_boxplot_clientes.show()

Esta célula define como os segmentos serão interpretados depois que o K-Means encontrar os grupos. O algoritmo retorna apenas números de grupos, como segmento 1, 2 e 3, mas esses números não têm significado de negócio sozinhos. Por isso, a função olha para os centroides e cria nomes como `alta fidelidade`, `risco de abandono` e `fidelidade média`. Essa parte é inferência humana, não uma verdade automática criada pelo modelo.

In [7]:
def interpretar_segmento(centroide_segmento: pd.Series) -> str:
    compras_mes = centroide_segmento['compras_mes']
    gasto_medio = centroide_segmento['gasto_medio_reais']
    dias_sem_comprar = centroide_segmento['dias_desde_ultima_compra']

    if compras_mes >= 8 and gasto_medio >= 180 and dias_sem_comprar <= 25:
        return 'alta fidelidade'

    return 'risco de abandono' if dias_sem_comprar >= 70 else 'fidelidade média'

Esta célula executa o fluxo principal da IA não supervisionada. Primeiro, os dados são normalizados com `MinMaxScaler`, porque compras, gasto e dias têm escalas diferentes. Depois, o `KMeans` encontra três grupos, o `PCA` reduz os dados para duas dimensões para visualização e o `silhouette_score` mede se os grupos ficaram razoavelmente separados. No final, a célula mostra os clientes segmentados e os centroides, que são os perfis médios de cada grupo.

In [9]:
def segmentar_clientes(base_clientes: pd.DataFrame) -> dict:
    colunas_entrada = ['compras_mes', 'gasto_medio_reais', 'dias_desde_ultima_compra']

    normalizador_clientes = MinMaxScaler()
    dados_normalizados = normalizador_clientes.fit_transform(base_clientes[colunas_entrada])

    modelo_segmentacao = KMeans(n_clusters=3, random_state=42, n_init=20)
    grupos_clientes = modelo_segmentacao.fit_predict(dados_normalizados)

    redutor_visualizacao = PCA(n_components=2)
    coordenadas_visualizacao = redutor_visualizacao.fit_transform(dados_normalizados)

    resultado_segmentacao = base_clientes.copy()
    resultado_segmentacao['segmento'] = grupos_clientes + 1
    resultado_segmentacao['pca_1'] = coordenadas_visualizacao[:, 0].round(5)
    resultado_segmentacao['pca_2'] = coordenadas_visualizacao[:, 1].round(5)

    centroides_segmentos = pd.DataFrame(
        normalizador_clientes.inverse_transform(modelo_segmentacao.cluster_centers_),
        columns=colunas_entrada,
    )
    centroides_segmentos['segmento'] = range(1, len(centroides_segmentos) + 1)
    centroides_segmentos['perfil'] = centroides_segmentos.apply(interpretar_segmento, axis=1)

    resultado_segmentacao = resultado_segmentacao.merge(
        centroides_segmentos[['segmento', 'perfil']],
        on='segmento',
        how='left',
    )

    return {
        'resultado_segmentacao': resultado_segmentacao,
        'centroides_segmentos': centroides_segmentos,
        'silhueta_segmentacao': silhouette_score(dados_normalizados, grupos_clientes),
        'variancia_pca': redutor_visualizacao.explained_variance_ratio_,
    }


resultado_clientes = segmentar_clientes(base_clientes)

print(f'Coeficiente de silhueta: {resultado_clientes["silhueta_segmentacao"]:.2f}')
display(resultado_clientes['resultado_segmentacao'].sort_values(['segmento', 'cliente']))
display(resultado_clientes['centroides_segmentos'].round(3))

Coeficiente de silhueta: 0.49


,cliente,compras_mes,gasto_medio_reais,dias_desde_ultima_compra,segmento,pca_1,pca_2,perfil
15,C016,6,114,47,1,0.02162,-0.06138,fidelidade média
16,C017,5,105,64,1,-0.10256,-0.02877,fidelidade média
19,C020,6,132,63,1,0.00803,0.03530,fidelidade média
24,C025,6,130,32,1,0.10020,-0.11632,fidelidade média
27,C028,3,91,69,1,-0.25294,-0.09901,fidelidade média
...,...,...,...,...,...,...,...,...
190,C191,8,163,3,3,0.36308,-0.14115,alta fidelidade
193,C194,9,173,40,3,0.32174,0.08766,alta fidelidade
195,C196,10,241,20,3,0.57338,0.10467,alta fidelidade
196,C197,9,210,33,3,0.41755,0.09410,alta fidelidade


,compras_mes,gasto_medio_reais,dias_desde_ultima_compra,segmento,perfil
0,4.700,110.571,54.414,1,fidelidade média
1,1.837,26.633,126.204,2,risco de abandono
2,8.568,217.519,22.185,3,alta fidelidade


Esta célula cria o gráfico principal da segmentação usando PCA. Cada ponto representa um cliente, a cor representa o perfil interpretado e o tamanho do ponto usa o gasto médio. O PCA não é o algoritmo de agrupamento, ele apenas transforma os dados para caberem em um gráfico de duas dimensões. Esse gráfico é ótimo para apresentação porque torna visual algo que normalmente estaria em três variáveis numéricas.

In [10]:
figura_segmentos = px.scatter(
    resultado_clientes['resultado_segmentacao'],
    x='pca_1',
    y='pca_2',
    color='perfil',
    symbol='segmento',
    size='gasto_medio_reais',
    hover_name='cliente',
    hover_data=['compras_mes', 'gasto_medio_reais', 'dias_desde_ultima_compra'],
    title='Segmentação de clientes visualizada com PCA',
    color_discrete_sequence=['#0ca678', '#f08c00', '#4263eb'],
)
figura_segmentos.update_layout(template='plotly_white')
figura_segmentos.show()

Esta célula resume quantos clientes ficaram em cada segmento. O gráfico de barras facilita explicar a distribuição dos grupos sem precisar ler a tabela linha por linha. Se um segmento tivesse clientes demais ou de menos, isso poderia indicar necessidade de revisar o valor de `k` ou investigar melhor os dados. Aqui ele funciona como uma leitura rápida da composição dos grupos.

In [11]:
distribuicao_segmentos = (
    resultado_clientes['resultado_segmentacao']
    .groupby(['segmento', 'perfil'])
    .size()
    .reset_index(name='quantidade_clientes')
    .sort_values('segmento')
)

figura_distribuicao = px.bar(
    distribuicao_segmentos,
    x='perfil',
    y='quantidade_clientes',
    color='perfil',
    text='quantidade_clientes',
    title='Quantidade de clientes por segmento',
    color_discrete_sequence=['#0ca678', '#f08c00', '#4263eb'],
)
figura_distribuicao.update_traces(textposition='outside')
figura_distribuicao.update_layout(template='plotly_white', 
                                showlegend=False,
                                height=600,
                                xaxis_title='Perfil do cliente',
                                yaxis_title='Quantidade de clientes',
                                )
figura_distribuicao.show()

Esta célula compara o perfil médio de cada segmento usando os centroides. Ela mostra como compras por mês, gasto médio e dias desde a última compra mudam de um grupo para outro. Esse gráfico ajuda a transformar o resultado técnico em interpretação de negócio, porque mostra por que um segmento parece estar em risco ou por que outro representa clientes de alto valor. É a ponte entre Machine Learning e decisão prática.

In [12]:
centroides_para_grafico = resultado_clientes['centroides_segmentos'].copy()
centroides_para_grafico['nome_segmento'] = centroides_para_grafico['segmento'].astype(str) + ' - ' + centroides_para_grafico['perfil']

figura_centroides = px.bar(
    centroides_para_grafico,
    x='nome_segmento',
    y=['compras_mes', 'gasto_medio_reais', 'dias_desde_ultima_compra'],
    barmode='group',
    title='Perfil médio de cada segmento',
)
figura_centroides.update_layout(template='plotly_white', xaxis_title='Segmento', yaxis_title='Valor médio')
figura_centroides.show()